# Shock Constructor — Penurunan Pariwisata Bali


## Sel 1 — Setup

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 1 — PERSIAPAN DATA                          ║
# ╚══════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sesuaikan path ini ke lokasi berkasmu di Colab / lokal
from google.colab import drive, files
drive.mount('/content/drive')

F_GEO  = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/02. Ekstraksi_Data.xlsx'
F_IRIO = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/01. Data_IRIO_17_Siap.xlsx'

print('Berkas proksi :', F_GEO)
print('Berkas IRIO   :', F_IRIO)


Mounted at /content/drive
Berkas proksi : /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/02. Ekstraksi_Data.xlsx
Berkas IRIO   : /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/01. Data_IRIO_17_Siap.xlsx


## Sel 2 — Konstanta sistem IRIO



In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 2 — KONSTANTA SISTEM IRIO                   ║
# ╚══════════════════════════════════════════════════╝

N_PROV  = 34
N_SEK   = 17
N_TOTAL = N_PROV * N_SEK          # 578

IDX_PROV_TARGET = 16
PROV_START = IDX_PROV_TARGET * N_SEK
PROV_END   = PROV_START + N_SEK

# Klasifikasi 17 sektor (S1-S17)
SEKTOR_17 = [
    'Pertanian, Kehutanan, dan Perikanan',
    'Pertambangan dan Penggalian',
    'Industri Pengolahan',
    'Pengadaan Listrik dan Gas',
    'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang',
    'Konstruksi',
    'Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor',
    'Transportasi dan Pergudangan',
    'Penyediaan Akomodasi dan Makan Minum',
    'Informasi dan Komunikasi',
    'Jasa Keuangan dan Asuransi',
    'Real Estate',
    'Jasa Perusahaan',
    'Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib',
    'Jasa Pendidikan',
    'Jasa Kesehatan dan Kegiatan Sosial',
    'Jasa Lainnya',
]
assert len(SEKTOR_17) == N_SEK
KODE_S = [f'S{i}' for i in range(1, N_SEK + 1)]

print(f'Sistem     : {N_PROV} provinsi x {N_SEK} sektor = {N_TOTAL} variabel')
print(f'Blok target: index {PROV_START+1} sampai {PROV_END}')


Sistem     : 34 provinsi x 17 sektor = 578 variabel
Blok target: index 273 sampai 289


## Sel 3 — Pemetaan kategori KBLI (proksi) -> sektor IRIO



In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 3 — PEMETAAN KATEGORI KBLI                  ║
# ╚══════════════════════════════════════════════════╝

# kode_kategori: (nama tampilan, [index 0-based sektor 17 yang diwakili])
KATEGORI = {
    'G':    ('Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor', [6]),
    'H':    ('Transportasi dan Pergudangan', [7]),
    'I':    ('Penyediaan Akomodasi dan Makan Minum', [8]),
    'JK':   ('Informasi&Komunikasi + Jasa Keuangan&Asuransi', [9, 10]),
    'L':    ('Real Estate', [11]),
    'MN':   ('Jasa Perusahaan', [12]),
    'P':    ('Jasa Pendidikan', [14]),
    'Q':    ('Jasa Kesehatan dan Kegiatan Sosial', [15]),
    'RSTU': ('Jasa Lainnya', [16]),
}
KODE_KAT = list(KATEGORI.keys())

# Normalisasi string kode dari file proksi ('J, K' -> 'JK', dst.)
def normalisasi_kode(s):
    return str(s).replace(' ', '').replace(',', '').strip().upper()

print('PEMETAAN KATEGORI -> SEKTOR IRIO')
for k in KODE_KAT:
    nama, idxs = KATEGORI[k]
    sektor_str = ', '.join(f'{KODE_S[i]} ({SEKTOR_17[i]})' for i in idxs)
    print(f'  {k:<6} -> {sektor_str}')


PEMETAAN KATEGORI -> SEKTOR IRIO
  G      -> S7 (Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor)
  H      -> S8 (Transportasi dan Pergudangan)
  I      -> S9 (Penyediaan Akomodasi dan Makan Minum)
  JK     -> S10 (Informasi dan Komunikasi), S11 (Jasa Keuangan dan Asuransi)
  L      -> S12 (Real Estate)
  MN     -> S13 (Jasa Perusahaan)
  P      -> S15 (Jasa Pendidikan)
  Q      -> S16 (Jasa Kesehatan dan Kegiatan Sosial)
  RSTU   -> S17 (Jasa Lainnya)


## Sel 4 — Perhitungan Perubahan



In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 4 — PARSING EKSTRAKSI_DATA.XLSX             ║
# ╚══════════════════════════════════════════════════╝

def buat_subsektor_key(kbli, subsektor):
    """
    Membuat kunci subsektor yang konsisten dengan KATEGORI
    pada SEL 3.
    """

    kbli_clean = normalisasi_kode(kbli)

    if subsektor is None or pd.isna(subsektor):
        return kbli_clean

    sub = str(subsektor).strip().lower()

    # KBLI H
    if kbli_clean == 'H':
        if 'rel' in sub:
            return 'H_rel'
        elif 'darat' in sub:
            return 'H_dar'
        elif 'laut' in sub or 'sungai' in sub:
            return 'H_lau'
        elif 'udara' in sub:
            return 'H_uda'
        elif 'gudang' in sub or 'pos' in sub:
            return 'H_gd'

    # KBLI I
    if kbli_clean == 'I':
        if 'akomodasi' in sub:
            return 'I_ako'
        elif 'makan' in sub or 'minum' in sub:
            return 'I_mkn'

    # KBLI lainnya
    return kbli_clean


def parse_ekstraksi(filepath):

    # Baca file
    df = pd.read_excel(filepath, header=1)

    print('Kolom asli:')
    print(df.columns.tolist())
    print()

    # Pastikan hanya mengambil lima kolom pertama
    df = df.iloc[:, :5].copy()

    df.columns = [
        'KBLI',
        'Subsektor',
        'Proksi',
        'Baseline',
        'Shock'
    ]

    # ────────────────────────────────────────────────
    # Bersihkan KBLI
    # ────────────────────────────────────────────────

    df['KBLI'] = (
        df['KBLI']
        .replace('-', np.nan)
        .ffill()
    )

    hasil = []

    kbli_saat_ini = None
    sub_saat_ini  = None

    for _, row in df.iterrows():

        # KBLI
        if pd.notna(row['KBLI']):
            kbli = normalisasi_kode(row['KBLI'])
        else:
            kbli = kbli_saat_ini

        # Jika KBLI berubah → reset subsektor
        if kbli != kbli_saat_ini:
            kbli_saat_ini = kbli
            sub_saat_ini = None

        # Subsektor baru
        if (
            pd.notna(row['Subsektor'])
            and str(row['Subsektor']).strip()
            not in ['', '-', 'nan']
        ):
            sub_saat_ini = str(row['Subsektor']).strip()

        # Ambil indikator yang lengkap saja
        if (
            pd.isna(row['Proksi'])
            or pd.isna(row['Baseline'])
            or pd.isna(row['Shock'])
        ):
            continue

        sub_key = buat_subsektor_key(
            kbli,
            sub_saat_ini
        )

        hasil.append({
            'KBLI': kbli,
            'Subsektor': sub_saat_ini,
            'Subsektor_Key': sub_key,
            'Proksi': str(row['Proksi']).strip(),
            'Baseline': float(row['Baseline']),
            'Shock': float(row['Shock'])
        })

    return pd.DataFrame(hasil)


# ──────────────────────────────────────────────────
# Jalankan parsing
# ──────────────────────────────────────────────────

df_proxy = parse_ekstraksi(F_GEO)

print()
print('✓ Parsing selesai')
print(f'  Total indikator : {len(df_proxy)}')
print(f'  KBLI unik       : {df_proxy["KBLI"].nunique()}')
print(f'  Subsektor unik  : {df_proxy["Subsektor_Key"].nunique()}')
print()

print('Ringkasan:')
print('─' * 75)

for key, grp in df_proxy.groupby(
    'Subsektor_Key',
    sort=False
):

    print(
        f'  {key:<10} | '
        f'KBLI {grp["KBLI"].iloc[0]:<5} | '
        f'n = {len(grp):>2} indikator | '
        f'{str(grp["Subsektor"].iloc[0])[:40]}'
    )

print('─' * 75)

Kolom asli:
[1, 2, 3, 4, 5]


✓ Parsing selesai
  Total indikator : 85
  KBLI unik       : 9
  Subsektor unik  : 14

Ringkasan:
───────────────────────────────────────────────────────────────────────────
  G          | KBLI G     | n =  6 indikator | Perdagangan Mobil, Sepeda Motor dan Repa
  H_rel      | KBLI H     | n =  3 indikator | Angkutan Rel
  H_dar      | KBLI H     | n =  7 indikator | Angkutan Darat
  H_lau      | KBLI H     | n =  6 indikator | Angkutan Laut; Angkutan Sungai Danau dan
  H_uda      | KBLI H     | n = 10 indikator | Angkutan Udara
  H_gd       | KBLI H     | n =  8 indikator | Pergudangan dan Jasa Penunjang Angkutan,
  I_mkn      | KBLI I     | n =  5 indikator | Penyediaan Makan Minum
  I_ako      | KBLI I     | n =  8 indikator | Penyediaan Akomodasi
  JK         | KBLI JK    | n =  5 indikator | Informasi & Komunikasi; Jasa Keuangan & 
  L          | KBLI L     | n =  5 indikator | Real Estat
  MN         | KBLI MN    | n =  5 indikator | Jasa Perusahaan (

# SEL 5 - HITUNG PERUBAHAN INDIKATOR

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 5 — HITUNG δ PER INDIKATOR                  ║
# ╚══════════════════════════════════════════════════╝

def hitung_delta(proksi, baseline, shock):

    p = str(proksi).lower()

    # ────────────────────────────────────────────────
    # 1. GOOGLE COMMUNITY MOBILITY
    # ────────────────────────────────────────────────
    if 'google' in p or 'mobility' in p:

        return (shock - baseline) / 100.0


    # ────────────────────────────────────────────────
    # 2. INDEKS DENGAN BASELINE NEGATIF
    # ────────────────────────────────────────────────
    if baseline < -1e-9:

        return (
            (shock - baseline)
            / abs(baseline)
        )


    # ────────────────────────────────────────────────
    # 3. INDIKATOR POSITIF
    # ────────────────────────────────────────────────
    if abs(baseline) > 1e-10:

        return (
            shock / baseline
        ) - 1.0


    # ────────────────────────────────────────────────
    # Baseline mendekati nol
    # ────────────────────────────────────────────────
    return 0.0


# ───────────────────────────────────────────────────
# HITUNG δ RAW
# ───────────────────────────────────────────────────

df_proxy['Delta_raw'] = df_proxy.apply(
    lambda r: hitung_delta(
        r['Proksi'],
        r['Baseline'],
        r['Shock']
    ),
    axis=1
)


# ───────────────────────────────────────────────────
# PENYESUAIAN ARAH GOOGLE MOBILITY RESIDENTIAL
# ───────────────────────────────────────────────────
#
# Residential memiliki arah interpretasi berlawanan:
#
# Residential ↑
# → waktu berada di rumah meningkat
# → mobilitas/aktivitas di luar rumah menurun
# → searah dengan shock
#
# Oleh karena itu, tanda Delta Residential dibalik.
#
# Delta_raw tetap dipertahankan sebagai perubahan asli.
# ───────────────────────────────────────────────────

mask_residential = (
    df_proxy['Proksi']
    .astype(str)
    .str.lower()
    .str.contains('residential')
)

df_proxy['Delta_arah'] = df_proxy['Delta_raw']

df_proxy.loc[
    mask_residential,
    'Delta_arah'
] = -df_proxy.loc[
    mask_residential,
    'Delta_raw'
]


# ───────────────────────────────────────────────────
# CLIPPING
# ───────────────────────────────────────────────────
#
# Setelah seluruh indikator memiliki arah yang sama:
#
# δ < 0  → berkontribusi terhadap shock
# δ ≥ 0  → tidak berkontribusi terhadap shock
#
# ───────────────────────────────────────────────────

df_proxy['Delta'] = (
    df_proxy['Delta_arah']
    .clip(upper=0)
)


# ───────────────────────────────────────────────────
# RINGKASAN
# ───────────────────────────────────────────────────

print('✓ δ berhasil dihitung')
print()

print(
    f'Seluruh δ ≤ 0 : '
    f'{(df_proxy["Delta"] <= 0).all()}'
)

print(
    f'Jumlah indikator : {len(df_proxy)}'
)

print(
    f'Jumlah δ = 0    : '
    f'{(df_proxy["Delta"] == 0).sum()}'
)

print(
    f'δ minimum       : '
    f'{df_proxy["Delta"].min():.6f}'
)

print()


# ───────────────────────────────────────────────────
# TAMPILKAN HASIL LENGKAP
# ───────────────────────────────────────────────────

print('HASIL δ PER INDIKATOR')
print('─' * 120)

print(
    df_proxy[
        [
            'KBLI',
            'Subsektor_Key',
            'Proksi',
            'Baseline',
            'Shock',
            'Delta_raw',
            'Delta_arah',
            'Delta'
        ]
    ].to_string(index=False)
)

print('─' * 120)

✓ δ berhasil dihitung

Seluruh δ ≤ 0 : True
Jumlah indikator : 85
Jumlah δ = 0    : 8
δ minimum       : -0.951361

HASIL δ PER INDIKATOR
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
KBLI Subsektor_Key                                                                                                    Proksi      Baseline         Shock  Delta_raw  Delta_arah     Delta
   G             G                                                                                    Night Time Light (NTL)  1.881374e-01      0.140867  -0.251257   -0.251257 -0.251257
   G             G                                                                            Land Surface Temperature (LST)  2.890410e+01     26.848416  -0.071121   -0.071121 -0.071121
   G             G                                                               Normalized Difference Built-up Index (NDBI) -3.198015e-01     -0.339155  -0.060517   -0.060517 -0.06051

# SEL 6 - AGREGASI INDIKATOR

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 6 — AGREGASI INDIKATOR → KATEGORI PARIWISATA ║
# ╚══════════════════════════════════════════════════╝


# ---------------------------------------------------
# 1. Fungsi Min-Max untuk shock
# ---------------------------------------------------
def minmax_shock(series):
    """
    Mengubah Delta menjadi skor 0–1.
    Delta paling kecil (shock paling besar) -> skor 1
    Delta paling besar (mendekati 0) -> skor 0
    """
    x = series.astype(float)

    xmin = x.min()
    xmax = x.max()

    if np.isclose(xmax, xmin):
        return pd.Series(0.0, index=x.index)

    return (xmax - x) / (xmax - xmin)


# ---------------------------------------------------
# 2. Normalisasi dalam masing-masing kelompok aktivitas
# ---------------------------------------------------
df_proxy['Score_Indikator'] = (
    df_proxy
    .groupby('Subsektor_Key')['Delta']
    .transform(minmax_shock)
)


# ---------------------------------------------------
# 3. Agregasi indikator → kelompok aktivitas
# ---------------------------------------------------
df_subsektor = (
    df_proxy
    .groupby(
        ['KBLI', 'Subsektor_Key', 'Subsektor'],
        dropna=False
    )
    .agg(
        Score_Subsektor=('Score_Indikator', 'mean'),
        Delta_Rata2=('Delta', 'mean'),
        N_Indikator=('Score_Indikator', 'count')
    )
    .reset_index()
)

print("=== SKOR KELOMPOK AKTIVITAS ===")
display(df_subsektor)


# ---------------------------------------------------
# 4. Agregasi kelompok aktivitas → kategori pariwisata
# ---------------------------------------------------
df_kategori = (
    df_subsektor
    .groupby('KBLI')
    .agg(
        Score_Kategori=('Score_Subsektor', 'mean'),
        Delta_Rata2=('Delta_Rata2', 'mean'),
        N_Subsektor=('Score_Subsektor', 'count'),
        N_Indikator=('N_Indikator', 'sum')
    )
    .reset_index()
)


# ---------------------------------------------------
# 5. Tambahkan nama kategori
# ---------------------------------------------------
df_kategori['Nama_Kategori'] = (
    df_kategori['KBLI']
    .map({k: v[0] for k, v in KATEGORI.items()})
)


# ---------------------------------------------------
# 6. Hitung intensitas shock δᵢ
# ---------------------------------------------------
df_kategori['delta_i'] = (
    -df_kategori['Score_Kategori']
)


# ---------------------------------------------------
# 7. Urutkan berdasarkan intensitas shock
# ---------------------------------------------------
df_kategori = (
    df_kategori
    .sort_values(
        'delta_i',
        ascending=True
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------
# 8. Tampilkan hasil
# ---------------------------------------------------
print("\n" + "="*75)
print("       INTENSITAS SHOCK 9 KATEGORI PARIWISATA")
print("="*75)

display(
    df_kategori[
        [
            'KBLI',
            'Nama_Kategori',
            'Score_Kategori',
            'delta_i',
            'N_Subsektor',
            'N_Indikator'
        ]
    ]
)

print("="*75)

print(
    f"Rentang δᵢ : "
    f"{df_kategori['delta_i'].min():.6f} "
    f"s.d. "
    f"{df_kategori['delta_i'].max():.6f}"
)

=== SKOR KELOMPOK AKTIVITAS ===


,KBLI,Subsektor_Key,Subsektor,Score_Subsektor,Delta_Rata2,N_Indikator
0,G,G,"Perdagangan Mobil, Sepeda Motor dan Reparasiny...",0.351949,-0.134691,6
1,H,H_dar,Angkutan Darat,0.289947,-0.194178,7
2,H,H_gd,"Pergudangan dan Jasa Penunjang Angkutan, Pos d...",0.265000,-0.177470,8
3,H,H_lau,Angkutan Laut; Angkutan Sungai Danau dan Penye...,0.539362,-0.462180,6
4,H,H_rel,Angkutan Rel,0.458393,-0.306986,3
5,H,H_uda,Angkutan Udara,0.513341,-0.488373,10
6,I,I_ako,Penyediaan Akomodasi,0.216334,-0.229940,8
7,I,I_mkn,Penyediaan Makan Minum,0.338292,-0.163366,5
8,JK,JK,Informasi & Komunikasi; Jasa Keuangan & Asuransi,0.401513,-0.126349,5
9,L,L,Real Estat,0.261383,-0.092298,5



       INTENSITAS SHOCK 9 KATEGORI PARIWISATA


,KBLI,Nama_Kategori,Score_Kategori,delta_i,N_Subsektor,N_Indikator
0,P,Jasa Pendidikan,0.486882,-0.486882,1,6
1,Q,Jasa Kesehatan dan Kegiatan Sosial,0.451277,-0.451277,1,4
2,H,Transportasi dan Pergudangan,0.413209,-0.413209,5,34
3,JK,Informasi&Komunikasi + Jasa Keuangan&Asuransi,0.401513,-0.401513,1,5
4,MN,Jasa Perusahaan,0.401513,-0.401513,1,5
5,G,Perdagangan Besar dan Eceran; Reparasi Mobil d...,0.351949,-0.351949,1,6
6,RSTU,Jasa Lainnya,0.349275,-0.349275,1,7
7,I,Penyediaan Akomodasi dan Makan Minum,0.277313,-0.277313,2,13
8,L,Real Estate,0.261383,-0.261383,1,5


Rentang δᵢ : -0.486882 s.d. -0.261383


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 6B — SIMPAN HASIL AGREGASI KE EXCEL       ║
# ╚══════════════════════════════════════════════════╝

F_OUT = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/03. Proporsi_Shock_Pariwisata_2020.xlsx'

with pd.ExcelWriter(F_OUT, engine='openpyxl') as writer:

    # Detail 85 indikator
    df_proxy.to_excel(
        writer,
        sheet_name='Indikator',
        index=False
    )

    # Hasil agregasi subsektor
    df_subsektor.to_excel(
        writer,
        sheet_name='Subsektor',
        index=False
    )

    # Hasil akhir 9 kategori + Pi
    df_kategori.to_excel(
        writer,
        sheet_name='Kategori_Pi',
        index=False
    )

print("✓ File berhasil disimpan!")
print("Lokasi:")
print(F_OUT)

✓ File berhasil disimpan!
Lokasi:
/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/03. Proporsi_Shock_Pariwisata_2020.xlsx


# SEL 8 — FD Bali 2025

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 8 — NORMALISASI INTENSITAS SHOCK            ║
# ╚══════════════════════════════════════════════════╝

def minmax_shock(x):
    """
    Mengubah Delta menjadi skor intensitas shock 0–1.

    Delta paling negatif → skor paling tinggi
    Delta mendekati 0    → skor paling rendah
    """

    x = pd.Series(x)

    xmin = x.min()
    xmax = x.max()

    # Jika seluruh nilai sama
    if np.isclose(xmax, xmin):
        return pd.Series(0.0, index=x.index)

    # Semakin negatif → skor semakin tinggi
    return (xmax - x) / (xmax - xmin)


# ---------------------------------------------------
# Normalisasi dalam masing-masing kelompok aktivitas
# ---------------------------------------------------

df_proxy['Skor_Indikator'] = (
    df_proxy
    .groupby('Subsektor_Key')['Delta']
    .transform(minmax_shock)
)


# ---------------------------------------------------
# Tampilkan hasil
# ---------------------------------------------------

kolom_normalisasi = [
    'KBLI',
    'Subsektor',
    'Proksi',
    'Delta_raw',
    'Delta_arah',
    'Delta',
    'Skor_Indikator'
]

display(
    df_proxy[kolom_normalisasi]
    .sort_values(['KBLI', 'Subsektor'])
    .reset_index(drop=True)
)

,KBLI,Subsektor,Proksi,Delta_raw,Delta_arah,Delta,Skor_Indikator
0,G,"Perdagangan Mobil, Sepeda Motor dan Reparasiny...",Night Time Light (NTL),-0.251257,-0.251257,-0.251257,0.656537
1,G,"Perdagangan Mobil, Sepeda Motor dan Reparasiny...",Land Surface Temperature (LST),-0.071121,-0.071121,-0.071121,0.185840
2,G,"Perdagangan Mobil, Sepeda Motor dan Reparasiny...",Normalized Difference Built-up Index (NDBI),-0.060517,-0.060517,-0.060517,0.158132
3,G,"Perdagangan Mobil, Sepeda Motor dan Reparasiny...",Built-Up Index (BUI),-0.042551,-0.042551,-0.042551,0.111187
4,G,"Perdagangan Mobil, Sepeda Motor dan Reparasiny...",Normalized Difference Vegetation Index (NDVI),0.034165,0.034165,0.000000,0.000000
...,...,...,...,...,...,...,...
80,RSTU,"Jasa Lainnya (seni, hiburan, rekreasi, jasa pe...",Normalized Difference Built-up Index (NDBI),-0.060517,-0.060517,-0.060517,0.108202
81,RSTU,"Jasa Lainnya (seni, hiburan, rekreasi, jasa pe...",Built-Up Index (BUI),-0.042551,-0.042551,-0.042551,0.076080
82,RSTU,"Jasa Lainnya (seni, hiburan, rekreasi, jasa pe...",Normalized Difference Vegetation Index (NDVI),0.034165,0.034165,0.000000,0.000000
83,RSTU,"Jasa Lainnya (seni, hiburan, rekreasi, jasa pe...",Google Community Mobility Report (Retail & Rec...,-0.382700,-0.382700,-0.382700,0.684248


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 9 — AGREGASI SKOR MENJADI δ SEKTOR          ║
# ╚══════════════════════════════════════════════════╝

# ---------------------------------------------------
# 1. Bobot indikator
# ---------------------------------------------------
# Equal weight dalam setiap subsektor

df_proxy['Bobot_Indikator'] = (
    1 /
    df_proxy.groupby('Subsektor_Key')['Proksi']
    .transform('count')
)


# ---------------------------------------------------
# 2. Weighted score per subsektor
# ---------------------------------------------------

df_subsektor = (
    df_proxy
    .groupby(['KBLI', 'Subsektor_Key', 'Subsektor'])
    .apply(
        lambda g: pd.Series({
            'Skor_Subsektor': (
                g['Bobot_Indikator'] *
                g['Skor_Indikator']
            ).sum()
        }),
        include_groups=False
    )
    .reset_index()
)


# ---------------------------------------------------
# 3. Agregasi subsektor → kategori
# ---------------------------------------------------

df_kategori = (
    df_subsektor
    .groupby('KBLI')
    .agg(
        Skor_Kategori=('Skor_Subsektor', 'mean'),
        Jumlah_Subsektor=('Subsektor_Key', 'count')
    )
    .reset_index()
)


# ---------------------------------------------------
# 4. Ubah menjadi koefisien perubahan δ
# ---------------------------------------------------

df_kategori['delta_i'] = -df_kategori['Skor_Kategori']


# ---------------------------------------------------
# 5. Tampilkan hasil
# ---------------------------------------------------

print("=" * 75)
print("        KOEFISIEN PERUBAHAN AKTIVITAS PARIWISATA")
print("=" * 75)

display(
    df_kategori[
        [
            'KBLI',
            'Jumlah_Subsektor',
            'Skor_Kategori',
            'delta_i'
        ]
    ]
)

        KOEFISIEN PERUBAHAN AKTIVITAS PARIWISATA


,KBLI,Jumlah_Subsektor,Skor_Kategori,delta_i
0,G,1,0.351949,-0.351949
1,H,5,0.413209,-0.413209
2,I,2,0.277313,-0.277313
3,JK,1,0.401513,-0.401513
4,L,1,0.261383,-0.261383
5,MN,1,0.401513,-0.401513
6,P,1,0.486882,-0.486882
7,Q,1,0.451277,-0.451277
8,RSTU,1,0.349275,-0.349275


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 10 — HITUNG SHOCK FINAL DEMAND (ΔFD)         ║
# ╚══════════════════════════════════════════════════╝

# ---------------------------------------------------
# 1. Baca Final Demand 2025
# ---------------------------------------------------

FD_raw = pd.read_excel(
    F_IRIO,
    sheet_name='FD25',
    header=None
).values.astype(float)

# Jumlah seluruh komponen Final Demand
fd_full = FD_raw.sum(axis=1)

assert len(fd_full) == N_TOTAL

print(f'✓ FD25: {FD_raw.shape}')
print(f'✓ Total FD nasional: Rp{fd_full.sum()/1e6:,.2f} triliun')


# ---------------------------------------------------
# 2. Ambil FD Bali
# ---------------------------------------------------

fd_bali = fd_full[PROV_START:PROV_END]

assert len(fd_bali) == N_SEK

print(f'✓ FD Bali: Rp{fd_bali.sum()/1e6:,.2f} triliun')


# ---------------------------------------------------
# 3. Mapping kategori pariwisata → sektor IRIO
# ---------------------------------------------------

MAP_KATEGORI = {
    'G':    [6],          # S7
    'H':    [7],          # S8
    'I':    [8],          # S9
    'JK':   [9, 10],      # S10 + S11
    'L':    [11],         # S12
    'MN':   [12],         # S13
    'P':    [14],         # S15
    'Q':    [15],         # S16
    'RSTU': [16],         # S17
}


# ---------------------------------------------------
# 4. Buat vektor shock ΔFD Bali
# ---------------------------------------------------

delta_fd_bali = np.zeros(N_SEK)
detail_delta_fd = []

for _, row in df_kategori.iterrows():

    kategori = row['KBLI']
    delta_i  = row['delta_i']

    if kategori not in MAP_KATEGORI:
        print(f'⚠ Kategori {kategori} tidak memiliki mapping.')
        continue

    for s_idx in MAP_KATEGORI[kategori]:

        fd_i = fd_bali[s_idx]

        # ΔFD = δᵢ × FDᵢ
        delta_fd_i = delta_i * fd_i

        delta_fd_bali[s_idx] = delta_fd_i

        detail_delta_fd.append({
            'Kategori': kategori,
            'Sektor': KODE_S[s_idx],
            'Nama_Sektor': SEKTOR_17[s_idx],
            'FD_2025_juta_Rp': fd_i,
            'delta_i': delta_i,
            'DeltaFD_juta_Rp': delta_fd_i,
            'DeltaFD_triliun_Rp': delta_fd_i / 1e6
        })


# ---------------------------------------------------
# 5. Tabel hasil
# ---------------------------------------------------

df_delta_fd = pd.DataFrame(detail_delta_fd)

print()
print("=" * 95)
print("              SHOCK FINAL DEMAND BALI 2025")
print("=" * 95)

display(
    df_delta_fd[
        [
            'Kategori',
            'Sektor',
            'Nama_Sektor',
            'FD_2025_juta_Rp',
            'delta_i',
            'DeltaFD_juta_Rp',
            'DeltaFD_triliun_Rp'
        ]
    ]
)


# ---------------------------------------------------
# 6. Ringkasan total shock
# ---------------------------------------------------

total_fd_pariwisata = df_delta_fd['FD_2025_juta_Rp'].sum()
total_delta_fd = df_delta_fd['DeltaFD_juta_Rp'].sum()

print()
print("=" * 70)
print("RINGKASAN")
print("=" * 70)

print(
    f"FD sektor terkait pariwisata : "
    f"Rp{total_fd_pariwisata/1e6:,.2f} triliun"
)

print(
    f"Total ΔFD / shock             : "
    f"Rp{total_delta_fd/1e6:,.2f} triliun"
)

print(
    f"Rasio ΔFD terhadap FD         : "
    f"{total_delta_fd/total_fd_pariwisata*100:.2f}%"
)


# ---------------------------------------------------
# 7. Validasi
# ---------------------------------------------------

assert len(df_delta_fd) == len(
    [s for v in MAP_KATEGORI.values() for s in v]
), "⚠ Jumlah sektor hasil mapping tidak sesuai."

assert (df_delta_fd['DeltaFD_juta_Rp'] <= 0).all(), \
    "⚠ Ada ΔFD positif."

print()
print("✓ Seluruh shock ΔFD bernilai negatif.")
print("✓ Semua kategori pariwisata berhasil dipetakan.")
print("✓ SEL 10 selesai.")

✓ FD25: (578, 1)
✓ Total FD nasional: Rp25,795.22 triliun
✓ FD Bali: Rp395.65 triliun

              SHOCK FINAL DEMAND BALI 2025


,Kategori,Sektor,Nama_Sektor,FD_2025_juta_Rp,delta_i,DeltaFD_juta_Rp,DeltaFD_triliun_Rp
0,G,S7,Perdagangan Besar dan Eceran; Reparasi Mobil d...,2.103360e+07,-0.351949,-7.402763e+06,-7.402763
1,H,S8,Transportasi dan Pergudangan,4.908411e+07,-0.413209,-2.028197e+07,-20.281974
2,I,S9,Penyediaan Akomodasi dan Makan Minum,9.943661e+07,-0.277313,-2.757510e+07,-27.575096
3,JK,S10,Informasi dan Komunikasi,1.232787e+07,-0.401513,-4.949795e+06,-4.949795
4,JK,S11,Jasa Keuangan dan Asuransi,7.965983e+06,-0.401513,-3.198443e+06,-3.198443
5,L,S12,Real Estate,9.846535e+06,-0.261383,-2.573721e+06,-2.573721
6,MN,S13,Jasa Perusahaan,1.353796e+06,-0.401513,-5.435663e+05,-0.543566
7,P,S15,Jasa Pendidikan,1.957252e+07,-0.486882,-9.529511e+06,-9.529511
8,Q,S16,Jasa Kesehatan dan Kegiatan Sosial,1.275292e+07,-0.451277,-5.755096e+06,-5.755096
9,RSTU,S17,Jasa Lainnya,7.392258e+06,-0.349275,-2.581930e+06,-2.581930



RINGKASAN
FD sektor terkait pariwisata : Rp240.77 triliun
Total ΔFD / shock             : Rp-84.39 triliun
Rasio ΔFD terhadap FD         : -35.05%

✓ Seluruh shock ΔFD bernilai negatif.
✓ Semua kategori pariwisata berhasil dipetakan.
✓ SEL 10 selesai.


# Sel 9 — Anchor kerugian (Rp) dan alokasi ke kategori Pariwisata



In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  SEL 9 — AGREGASI FD KATEGORI PARIWISATA         ║
# ╚══════════════════════════════════════════════════╝

# Mapping kategori pariwisata → sektor IRIO
MAP_FD = {
    'G':    [6],          # S7
    'H':    [7],          # S8
    'I':    [8],          # S9
    'JK':   [9, 10],      # S10 + S11
    'L':    [11],         # S12
    'MN':   [12],         # S13
    'P':    [14],         # S15
    'Q':    [15],         # S16
    'RSTU': [16],         # S17
}

# Ambil FD berdasarkan indeks sektor
fd_kategori = {}

for kat, sektor_idx in MAP_FD.items():
    fd_kategori[kat] = fd_bali[sektor_idx].sum()

df_fd_kategori = pd.DataFrame({
    'KBLI': list(fd_kategori.keys()),
    'FD_2025': list(fd_kategori.values())
})

df_fd_kategori['Proporsi_FD'] = (
    df_fd_kategori['FD_2025']
    / df_fd_kategori['FD_2025'].sum()
)

display(df_fd_kategori)

,KBLI,FD_2025,Proporsi_FD
0,G,2.103360e+07,0.087361
1,H,4.908411e+07,0.203866
2,I,9.943661e+07,0.413001
3,JK,2.029385e+07,0.084289
4,L,9.846535e+06,0.040897
5,MN,1.353796e+06,0.005623
6,P,1.957252e+07,0.081293
7,Q,1.275292e+07,0.052968
8,RSTU,7.392258e+06,0.030703


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║ SEL 10 — SIMPAN SHOCK FINAL DEMAND UNTUK SIMULASI IRIO ║
# ╚══════════════════════════════════════════════════════════╝

# ============================================================
# 1. HITUNG FD 2025
# ============================================================

FD_raw = pd.read_excel(
    F_IRIO,
    sheet_name='FD25',
    header=None
).values.astype(float)

fd_full = FD_raw.sum(axis=1)

assert len(fd_full) == N_TOTAL

fd_bali = fd_full[PROV_START:PROV_END]

print(f'✓ FD25 nasional : Rp{fd_full.sum()/1e6:,.2f} triliun')
print(f'✓ FD25 Bali     : Rp{fd_bali.sum()/1e6:,.2f} triliun')


# ============================================================
# 2. MAPPING KATEGORI → SEKTOR IRIO
# ============================================================

MAP_KATEGORI = {
    'G':    [6],       # S7
    'H':    [7],       # S8
    'I':    [8],       # S9
    'JK':   [9, 10],   # S10, S11
    'L':    [11],      # S12
    'MN':   [12],      # S13
    'P':    [14],      # S15
    'Q':    [15],      # S16
    'RSTU': [16],      # S17
}


# ============================================================
# 3. BENTUK SHOCK ΔFD BALI (17 × 1)
# ============================================================

delta_fd_bali = np.zeros(N_SEK)

detail_delta_fd = []

for _, row in df_kategori.iterrows():

    kategori = row['KBLI']
    delta_i  = row['delta_i']

    if kategori not in MAP_KATEGORI:
        print(f'⚠ Kategori {kategori} tidak memiliki mapping.')
        continue

    for s_idx in MAP_KATEGORI[kategori]:

        fd_i = fd_bali[s_idx]

        # Shock final demand
        delta_fd_i = delta_i * fd_i

        delta_fd_bali[s_idx] = delta_fd_i

        detail_delta_fd.append({
            'Kategori': kategori,
            'Sektor': KODE_S[s_idx],
            'Nama_Sektor': SEKTOR_17[s_idx],
            'FD_2025_juta_Rp': fd_i,
            'delta_i': delta_i,
            'DeltaFD_juta_Rp': delta_fd_i,
            'DeltaFD_triliun_Rp': delta_fd_i / 1e6
        })


df_delta_fd = pd.DataFrame(detail_delta_fd)


# ============================================================
# 4. TABEL 17 SEKTOR BALI
# ============================================================

df_shock_bali = pd.DataFrame({
    'Sektor': KODE_S,
    'Nama_Sektor': SEKTOR_17,
    'DeltaFD_juta_Rp': delta_fd_bali
})

df_shock_bali['DeltaFD_triliun_Rp'] = (
    df_shock_bali['DeltaFD_juta_Rp'] / 1e6
)


# ============================================================
# 5. BENTUK VEKTOR SHOCK NASIONAL 578 × 1
# ============================================================
#
# Urutan harus sama persis dengan matriks IRIO:
#
# Provinsi 1 : S1 ... S17
# Provinsi 2 : S1 ... S17
# ...
# Provinsi 17 (Bali) : S1 ... S17
# ...
# Provinsi 34 : S1 ... S17
#
# Shock hanya diberikan kepada Bali.
# ============================================================

delta_fd_578 = np.zeros(N_TOTAL)

delta_fd_578[
    PROV_START:PROV_END
] = delta_fd_bali


# Validasi dimensi
assert len(delta_fd_578) == 578
assert len(delta_fd_bali) == 17

# Validasi hanya Bali yang terkena shock
assert np.count_nonzero(delta_fd_578) == np.count_nonzero(delta_fd_bali)


# ============================================================
# 6. TABEL SHOCK 578 UNIT
# ============================================================

df_shock_578 = []

for p_idx in range(N_PROV):

    for s_idx in range(N_SEK):

        idx = p_idx * N_SEK + s_idx

        df_shock_578.append({
            'Provinsi_ID': p_idx + 1,
            'Sektor_ID': s_idx + 1,
            'Index_IRIO': idx,
            'Sektor': KODE_S[s_idx],
            'Nama_Sektor': SEKTOR_17[s_idx],
            'DeltaFD_juta_Rp': delta_fd_578[idx],
            'DeltaFD_triliun_Rp': delta_fd_578[idx] / 1e6
        })

df_shock_578 = pd.DataFrame(df_shock_578)


# ============================================================
# 7. RINGKASAN HASIL
# ============================================================

total_fd_pariwisata = (
    df_delta_fd['FD_2025_juta_Rp'].sum()
)

total_delta_fd = (
    df_delta_fd['DeltaFD_juta_Rp'].sum()
)

print()
print('=' * 75)
print('              SHOCK FINAL DEMAND BALI 2025')
print('=' * 75)

display(
    df_delta_fd[
        [
            'Kategori',
            'Sektor',
            'Nama_Sektor',
            'FD_2025_juta_Rp',
            'delta_i',
            'DeltaFD_juta_Rp',
            'DeltaFD_triliun_Rp'
        ]
    ]
)

print('=' * 75)
print('RINGKASAN')
print('=' * 75)

print(
    f'FD sektor terkait pariwisata : '
    f'Rp{total_fd_pariwisata/1e6:,.2f} triliun'
)

print(
    f'Total ΔFD / shock             : '
    f'Rp{total_delta_fd/1e6:,.2f} triliun'
)

print(
    f'Rasio ΔFD terhadap FD         : '
    f'{total_delta_fd/total_fd_pariwisata*100:.2f}%'
)


# ============================================================
# 8. VALIDASI
# ============================================================

assert (df_shock_bali['DeltaFD_juta_Rp'] <= 0).all()
assert (df_shock_578['DeltaFD_juta_Rp'] <= 0).all()

# Total shock 17 sektor Bali harus sama dengan
# total shock pada 578 unit nasional
assert np.isclose(
    df_shock_bali['DeltaFD_juta_Rp'].sum(),
    df_shock_578['DeltaFD_juta_Rp'].sum()
)

print()
print('✓ Seluruh shock ΔFD bernilai negatif.')
print('✓ Vektor Bali berukuran 17 × 1.')
print('✓ Vektor nasional berukuran 578 × 1.')
print('✓ Total shock Bali = total shock pada vektor nasional.')

✓ FD25 nasional : Rp25,795.22 triliun
✓ FD25 Bali     : Rp395.65 triliun

              SHOCK FINAL DEMAND BALI 2025


,Kategori,Sektor,Nama_Sektor,FD_2025_juta_Rp,delta_i,DeltaFD_juta_Rp,DeltaFD_triliun_Rp
0,G,S7,Perdagangan Besar dan Eceran; Reparasi Mobil d...,2.103360e+07,-0.351949,-7.402763e+06,-7.402763
1,H,S8,Transportasi dan Pergudangan,4.908411e+07,-0.413209,-2.028197e+07,-20.281974
2,I,S9,Penyediaan Akomodasi dan Makan Minum,9.943661e+07,-0.277313,-2.757510e+07,-27.575096
3,JK,S10,Informasi dan Komunikasi,1.232787e+07,-0.401513,-4.949795e+06,-4.949795
4,JK,S11,Jasa Keuangan dan Asuransi,7.965983e+06,-0.401513,-3.198443e+06,-3.198443
5,L,S12,Real Estate,9.846535e+06,-0.261383,-2.573721e+06,-2.573721
6,MN,S13,Jasa Perusahaan,1.353796e+06,-0.401513,-5.435663e+05,-0.543566
7,P,S15,Jasa Pendidikan,1.957252e+07,-0.486882,-9.529511e+06,-9.529511
8,Q,S16,Jasa Kesehatan dan Kegiatan Sosial,1.275292e+07,-0.451277,-5.755096e+06,-5.755096
9,RSTU,S17,Jasa Lainnya,7.392258e+06,-0.349275,-2.581930e+06,-2.581930


RINGKASAN
FD sektor terkait pariwisata : Rp240.77 triliun
Total ΔFD / shock             : Rp-84.39 triliun
Rasio ΔFD terhadap FD         : -35.05%

✓ Seluruh shock ΔFD bernilai negatif.
✓ Vektor Bali berukuran 17 × 1.
✓ Vektor nasional berukuran 578 × 1.
✓ Total shock Bali = total shock pada vektor nasional.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║ SEL 11 — SIMPAN OUTPUT SHOCK UNTUK SIMULASI IRIO       ║
# ╚══════════════════════════════════════════════════════════╝

F_OUT_SHOCK = (
    '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/'
    '[01] Skripsi Atikah/Data/'
    '05. Hasil_Shock_FinalDemand_Bali_2025.xlsx'
)


with pd.ExcelWriter(F_OUT_SHOCK, engine='openpyxl') as writer:

    # Detail 10 sektor yang terkena shock
    df_delta_fd.to_excel(
        writer,
        sheet_name='Shock_10_Sektor',
        index=False
    )

    # Vektor shock Bali 17 sektor
    df_shock_bali.to_excel(
        writer,
        sheet_name='Shock_Bali_17',
        index=False
    )

    # Vektor shock nasional 578 unit
    df_shock_578.to_excel(
        writer,
        sheet_name='Shock_Full_578',
        index=False
    )


print()
print('=' * 75)
print('FILE SHOCK BERHASIL DISIMPAN')
print('=' * 75)
print(F_OUT_SHOCK)


FILE SHOCK BERHASIL DISIMPAN
/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/05. Hasil_Shock_FinalDemand_Bali_2025.xlsx


In [ ]:
# ---------------------------------------------------
# Simpan juga dalam format .npy dan .csv untuk sel simulasi IRIO berikutnya
# ---------------------------------------------------

F_DELTAF_OUT = (
    '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/'
    '[01] Skripsi Atikah/Data/'
    '07. delta_f.npy'
)

F_DETAIL_OUT = (
    '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/'
    '[01] Skripsi Atikah/Data/'
    '06. delta_f_detail.csv'
)

np.save(F_DELTAF_OUT, delta_fd_578)
df_shock_578.to_csv(F_DETAIL_OUT, index=False)

print()
print('=' * 75)
print('FILE UNTUK SIMULASI LEONTIEF BERHASIL DISIMPAN')
print('=' * 75)
print('✓', F_DELTAF_OUT, '| shape:', delta_fd_578.shape)
print('✓', F_DETAIL_OUT, '| shape:', df_shock_578.shape)


FILE UNTUK SIMULASI LEONTIEF BERHASIL DISIMPAN
✓ /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/07. delta_f.npy | shape: (578,)
✓ /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/06. delta_f_detail.csv | shape: (578, 7)
